# 📊 AiStock 可视化分析 (Plotly 可视化版)

## 分析目标 + 交互式可视化展示
- ✅ 动态价格分布：价格区间 + 盈亏比散点图（交互式筛选）
- ✅ 板块配置分析：板块权重饼图 + 盈亏对比条形图（交互式）
- ✅ 风险收益分析：风险矩阵散点图 + 四象限标注（交互式）
- ✅ 组合表现可视化：净值曲线 + 风险指标仪表板（交互式）

## Plotly 高级特性：散点矩阵、桑基图、瀑布图、交互式筛选器

In [ ]:
# 环境设置 + Plotly 高级可视化配置 + 中文支持准备 + 分析工具函数
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Plotly 主题 + 中文配置（可选：安装中文字体后启用）
import plotly.io as pio
pio.templates.default = "plotly_white"

# 添加项目根目录到路径 + 模拟数据生成器（演示用）
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 模拟分析数据生成器（用于可视化演示）
def generate_analysis_data(n_stocks=18):
    """生成模拟分析数据用于可视化"""
    np.random.seed(42)
    
    stocks = [
        ('600938', '中国海油', '油气开采'),
        ('601857', '中国石油', '油气开采'),
        ('600256', '广汇能源', 'LNG'),
        ('600803', '新奥股份', 'LNG'),
        ('601808', '中海油服', '油服'),
        ('002353', '杰瑞股份', '油服'),
        ('601088', '中国神华', '煤炭化工'),
        ('600426', '华鲁恒升', '煤炭化工'),
        ('600406', '国电南瑞', '特高压'),
        ('000400', '许继电气', '特高压'),
        ('300750', '宁德时代', '新能源'),
        ('300274', '阳光电源', '新能源'),
        ('601899', '紫金矿业', '黄金'),
        ('600489', '中金黄金', '黄金'),
        ('600760', '中航沈飞', '军工'),
        ('002179', '中航光电', '军工'),
        ('603019', '中科曙光', '政策方向'),
        ('600536', '中国软件', '政策方向')
    ]
    
    data = []
    for code, name, sector in stocks[:n_stocks]:
        data.append({
            'code': code,
            'name': name,
            'sector': sector,
            'current_price': np.random.uniform(20, 100),
            'entry_price': np.random.uniform(18, 95),
            'stop_loss': np.random.uniform(15, 90),
            'target_price': np.random.uniform(25, 120),
            'profit_loss_ratio': np.random.uniform(1.5, 4.5),
            'fundamental_score': np.random.uniform(50, 95),
            'macro_factor': np.random.uniform(0.95, 1.10),
            'volatility': np.random.uniform(0.15, 0.45),
            'recommendation': np.random.choice(['强烈推荐', '推荐', '观望', '谨慎'], 
                                              p=[0.2, 0.4, 0.3, 0.1])
        })
    
    return pd.DataFrame(data)

print("✅ 环境设置完成 | Plotly 高级可视化已启用")

In [ ]:
# 导入系统模块 + 生成模拟分析数据（演示）
from base_services.config_service import ConfigService
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine

# 生成模拟分析数据（实际应从系统获取）
analysis_df = generate_analysis_data(n_stocks=18)

# 计算衍生指标用于分析
analysis_df['potential_profit'] = analysis_df['target_price'] - analysis_df['entry_price']
analysis_df['potential_loss'] = analysis_df['entry_price'] - analysis_df['stop_loss']
analysis_df['risk_reward'] = analysis_df['potential_profit'] / analysis_df['potential_loss']

print(f"✅ 生成分析数据：{len(analysis_df)}只标的")
print(f"📊 数据概览：\n{analysis_df[['code', 'name', 'sector', 'recommendation']].head()}")

## 1️⃣ 动态价格分布 + 交互式散点图矩阵

In [ ]:
# Plotly 散点图矩阵（多维度关联分析 + 交互式筛选）
fig = px.scatter_matrix(
    analysis_df,
    dimensions=['fundamental_score', 'profit_loss_ratio', 'macro_factor', 'volatility'],
    color='recommendation',
    color_discrete_map={
        '强烈推荐': 'green',
        '推荐': 'blue',
        '观望': 'orange',
        '谨慎': 'red'
    },
    title='🔗 多维度关联分析（散点图矩阵）',
    labels={
        'fundamental_score': '基本面评分',
        'profit_loss_ratio': '盈亏比',
        'macro_factor': '宏观系数',
        'volatility': '波动率'
    },
    hover_data=['code', 'name', 'sector']
)

# 更新布局 + 交互式设置（对角线直方图 + 悬停提示）
fig.update_layout(
    height=600,
    hovermode='closest',
    legend=dict(orientation='h', yanchor='bottom', y=-0.1, xanchor='center', x=0.5),
    title_font_size=14
)

# 更新对角线为直方图（显示分布）
fig.update_traces(
    diagonal_visible=True,
    hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]})<br>板块：%{customdata[2]}<br>基本面：%{x:.1f}<br>盈亏比：%{y:.2f}<extra></extra>',
    customdata=analysis_df[['code', 'name', 'sector']].values
)

fig.show()

In [ ]:
# Plotly 价格区间瀑布图（交互式展示盈亏构成）
# 准备瀑布图数据（按盈亏比排序）
sorted_df = analysis_df.sort_values('profit_loss_ratio', ascending=False)

# 创建瀑布图数据（Plotly 瀑布图需要特殊格式）
waterfall_data = []

# 添加初始值（基准）
waterfall_data.append({
    'x': ['基准'],
    'y': [100],
    'measure': ['absolute'],
    'text': ['100'],
    'textposition': 'outside'
})

# 添加各标的盈亏贡献（相对值）
for _, row in sorted_df.head(10).iterrows():  # Top 10
    contribution = (row['profit_loss_ratio'] - 2.5) * 10  # 简化贡献计算
    waterfall_data.append({
        'x': [row['name']],
        'y': [contribution],
        'measure': ['relative'],
        'text': [f"{row['profit_loss_ratio']:.1f}x"],
        'textposition': 'outside',
        'marker': {'color': 'red' if contribution > 0 else 'green'}
    })

# 添加总计（绝对值）
total = 100 + sum((r['profit_loss_ratio'] - 2.5) * 10 for _, r in sorted_df.head(10).iterrows())
waterfall_data.append({
    'x': ['组合'],
    'y': [total],
    'measure': ['total'],
    'text': [f"{total:.1f}"],
    'textposition': 'outside'
})

# 创建瀑布图（交互式）
fig = go.Figure()

for item in waterfall_data:
    fig.add_trace(go.Waterfall(
        x=item['x'],
        y=item['y'],
        measure=item['measure'],
        text=item['text'],
        textposition=item['textposition'],
        marker=item.get('marker', {}),
        connector=dict(line=dict(color='gray', width=1))
    ))

fig.update_layout(
    title='💰 标的盈亏贡献瀑布图（Top 10）',
    height=500,
    hovermode='x unified',
    xaxis_title='标的',
    yaxis_title='贡献值',
    xaxis_tickangle=-45
)

fig.show()

## 2️⃣ 板块配置分析 + 交互式饼图 + 条形图

In [ ]:
# Plotly 板块分布饼图（交互式 + 悬停详情）
sector_summary = analysis_df.groupby('sector').agg({
    'code': 'count',
    'profit_loss_ratio': 'mean',
    'fundamental_score': 'mean'
}).reset_index().rename(columns={'code': 'count'})

# 创建交互式板块分布饼图（环形 + 悬停）
fig = px.pie(
    sector_summary,
    values='count',
    names='sector',
    title='📊 板块配置分布',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 添加交互式悬停提示（显示详细数据）
fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>标的数：%{value} 只<br>平均盈亏比：%{customdata[0]:.2f}x<br>平均评分：%{customdata[1]:.1f}<extra></extra>',
    customdata=sector_summary[['profit_loss_ratio', 'fundamental_score']].values
)

fig.update_layout(
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5)
)

fig.show()

In [ ]:
# Plotly 板块盈亏对比条形图（交互式 + 颜色编码）
# 按板块汇总盈亏比 + 评分（用于对比）
sector_perf = analysis_df.groupby('sector').agg({
    'profit_loss_ratio': ['mean', 'std'],
    'fundamental_score': 'mean',
    'code': 'count'
}).round(2).reset_index()
sector_perf.columns = ['sector', 'avg_pl', 'std_pl', 'avg_score', 'count']

# 创建交互式条形图（盈亏比 + 评分双指标）
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['平均盈亏比', '平均基本面评分'],
    specs=[[{'secondary_y': False}, {'secondary_y': False}]]
)

# 左图：平均盈亏比（带误差线）
fig.add_trace(go.Bar(
    x=sector_perf['sector'],
    y=sector_perf['avg_pl'],
    error_y=dict(type='data', array=sector_perf['std_pl'], visible=True),
    name='盈亏比',
    marker_color='skyblue',
    text=sector_perf['avg_pl'].apply(lambda x: f'{x:.1f}x'),
    textposition='auto'
), row=1, col=1)

# 右图：平均基本面评分（带计数标注）
fig.add_trace(go.Bar(
    x=sector_perf['sector'],
    y=sector_perf['avg_score'],
    name='基本面评分',
    marker_color='coral',
    text=sector_perf['avg_score'].apply(lambda x: f'{x:.0f}'),
    textposition='auto'
), row=1, col=2)

# 更新布局 + 交互式设置（悬停提示 + 图例）
fig.update_layout(
    title='📈 板块绩效对比分析',
    height=450,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5)
)

# 更新坐标轴标签 + 旋转板块名称
fig.update_xaxes(title_text='板块', tickangle=-45, row=1, col=1)
fig.update_xaxes(title_text='板块', tickangle=-45, row=1, col=2)
fig.update_yaxes(title_text='盈亏比 (x)', row=1, col=1)
fig.update_yaxes(title_text='评分', range=[0, 100], row=1, col=2)

fig.show()

## 3️⃣ 风险收益分析 + 交互式风险矩阵

In [ ]:
# Plotly 风险矩阵散点图（波动率×盈亏比 + 四象限分析）
# 创建交互式风险矩阵（气泡图 + 颜色编码）
fig = px.scatter(
    analysis_df,
    x='volatility',
    y='profit_loss_ratio',
    size='fundamental_score',
    color='recommendation',
    hover_name='name',
    title='🎯 风险收益矩阵（波动率×盈亏比）',
    labels={'volatility': '风险（波动率）', 'profit_loss_ratio': '收益（盈亏比）'},
    color_discrete_map={
        '强烈推荐': 'green',
        '推荐': 'blue',
        '观望': 'orange',
        '谨慎': 'red'
    },
    size_max=40,
    opacity=0.8
)

# 添加参考线（风险收益平衡线 + 阈值线）
fig.add_hline(y=2.5, line_dash='dash', line_color='blue',
              annotation_text='盈亏比阈值: 2.5x', annotation_position='top right')
fig.add_vline(x=0.30, line_dash='dash', line_color='orange',
              annotation_text='波动率阈值: 30%', annotation_position='bottom right')

# 添加四象限标注（交互式）
quadrants = [
    {'x': 0.20, 'y': 3.5, 'text': '✅ 低风险高收益', 'color': 'lightgreen'},
    {'x': 0.40, 'y': 3.5, 'text': '⚠️ 高风险高收益', 'color': 'lightyellow'},
    {'x': 0.20, 'y': 1.5, 'text': '⚪ 低风险低收益', 'color': 'lightgray'},
    {'x': 0.40, 'y': 1.5, 'text': '❌ 高风险低收益', 'color': 'lightcoral'}
]

for q in quadrants:
    fig.add_annotation(
        x=q['x'], y=q['y'],
        text=q['text'],
        showarrow=False,
        bgcolor=q['color'],
        font=dict(size=9),
        borderwidth=1,
        bordercolor='gray'
    )

# 更新布局 + 交互式设置（悬停提示 + 图例筛选）
fig.update_layout(
    height=500,
    hovermode='closest',
    xaxis_title='风险（波动率）',
    yaxis_title='收益（盈亏比）',
    legend_title='投资建议',
    xaxis_range=[0, 0.6],
    yaxis_range=[0, 5]
)

# 更新悬停提示（显示详细信息）
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>代码：%{customdata[0]}<br>板块：%{customdata[1]}<br>风险：%{x:.1%}<br>收益：%{y:.1f}x<br>评分：%{marker.size:.0f}<extra></extra>',
    customdata=analysis_df[['code', 'sector']].values
)

fig.show()

In [ ]:
# Plotly 风险指标仪表板（多指标卡片 + 交互式）
# 计算组合风险指标（模拟）
portfolio_metrics = {
    '夏普比率': np.random.uniform(0.8, 2.2),
    '最大回撤': -np.random.uniform(0.08, 0.22),
    '年化波动率': np.random.uniform(0.12, 0.28),
    '胜率': np.random.uniform(0.52, 0.68),
    '平均盈亏比': analysis_df['profit_loss_ratio'].mean(),
    '推荐标的占比': (analysis_df['recommendation'].isin(['强烈推荐', '推荐'])).mean()
}

# 创建交互式风险指标仪表板（多指标卡片）
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(portfolio_metrics.keys()),
    specs=[[{'type': 'indicator'}]*3, [{'type': 'indicator'}]*3]
)

# 添加各指标仪表（动态配置阈值）
thresholds = {
    '夏普比率': {'ref': 1.0, 'range': [0, 3], 'color': 'darkblue'},
    '最大回撤': {'ref': -0.15, 'range': [-0.3, 0], 'color': 'darkred'},
    '年化波动率': {'ref': 0.20, 'range': [0, 0.4], 'color': 'darkorange'},
    '胜率': {'ref': 0.55, 'range': [0, 1], 'color': 'darkgreen'},
    '平均盈亏比': {'ref': 2.5, 'range': [0, 5], 'color': 'darkblue'},
    '推荐标的占比': {'ref': 0.60, 'range': [0, 1], 'color': 'darkgreen'}
}

for idx, (metric, value) in enumerate(portfolio_metrics.items()):
    row = idx // 3 + 1
    col = idx % 3 + 1
    thresh = thresholds[metric]
    
    fig.add_trace(go.Indicator(
        mode='gauge+number+delta',
        value=value,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': metric},
        delta={'reference': thresh['ref']},
        gauge={'axis': {'range': thresh['range']}, 'bar': {'color': thresh['color']}}
    ), row=row, col=col)

fig.update_layout(
    title='🎯 组合风险指标仪表板',
    height=500,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

## 4️⃣ 组合表现可视化 + 交互式净值曲线

In [ ]:
# Plotly 净值曲线对比图（组合 vs 基准 + 交互式）
# 生成模拟净值数据（实际应从组合跟踪器获取）
dates = pd.date_range('2026-01-01', periods=90, freq='D')
np.random.seed(42)

# 模拟组合净值（带随机波动 + 趋势）
portfolio_returns = np.random.normal(0.0006, 0.012, len(dates))  # 日均收益 0.06%, 波动 1.2%
portfolio_nav = 1000000 * np.cumprod(1 + portfolio_returns)

# 模拟基准指数（沪深 300 模拟）
benchmark_returns = np.random.normal(0.0003, 0.010, len(dates))
benchmark_nav = 1000000 * np.cumprod(1 + benchmark_returns)

# 创建交互式净值对比图（双曲线 + 超额收益区域）
fig = go.Figure()

# 添加组合净值曲线（蓝色实线）
fig.add_trace(go.Scatter(
    x=dates, y=portfolio_nav,
    name='组合净值',
    line=dict(color='blue', width=2),
    hovertemplate='<b>组合</b><br>日期：%{x}<br>净值：¥%{y:,.0f}<br>累计收益：%{customdata:.1%}<extra></extra>',
    customdata=(portfolio_nav - 1000000) / 1000000
))

# 添加基准曲线（灰色虚线）
fig.add_trace(go.Scatter(
    x=dates, y=benchmark_nav,
    name='基准 (沪深 300)',
    line=dict(color='gray', width=1, dash='dash'),
    hovertemplate='<b>基准</b><br>日期：%{x}<br>净值：¥%{y:,.0f}<br>累计收益：%{customdata:.1%}<extra></extra>',
    customdata=(benchmark_nav - 1000000) / 1000000
))

# 添加超额收益区域（组合 - 基准）
excess = portfolio_nav - benchmark_nav
fig.add_trace(go.Scatter(
    x=dates, y=excess,
    name='超额收益',
    mode='lines',
    line=dict(width=0),
    fill='tozeroy',
    fillcolor='rgba(0,128,0,0.2)' if excess.iloc[-1] > 0 else 'rgba(255,0,0,0.2)',
    showlegend=True,
    hovertemplate='超额收益：¥%{y:,.0f}<extra></extra>'
))

# 添加最大回撤标注（模拟计算）
cummax = np.maximum.accumulate(portfolio_nav)
drawdown = (portfolio_nav - cummax) / cummax
max_dd = drawdown.min()
max_dd_date = dates[drawdown.idxmin()]

fig.add_annotation(
    x=max_dd_date,
    y=portfolio_nav.iloc[drawdown.idxmin()],
    text=f'最大回撤：{max_dd:.1%}',
    showarrow=True,
    arrowhead=2,
    bgcolor='lightcoral' if max_dd < -0.15 else 'lightgreen',
    font=dict(color='white' if abs(max_dd) > 0.15 else 'black')
)

# 更新布局 + 交互式设置（范围滑块 + 图例筛选 + 悬停提示）
fig.update_layout(
    title='📈 组合表现对比（净值曲线）',
    xaxis_title='日期',
    yaxis_title='净值 (元)',
    yaxis_tickformat=',.0f',
    height=450,
    hovermode='x unified',
    xaxis_rangeslider_visible=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5)
)

fig.show()

In [ ]:
# Plotly 标的推荐列表（交互式表格 + 排序筛选）
# 准备推荐标的数据（按盈亏比 + 评分综合排序）
analysis_df['composite_score'] = (
    analysis_df['profit_loss_ratio'] * 0.6 + 
    (analysis_df['fundamental_score'] / 100) * 10 * 0.4
)
top_recommendations = analysis_df[
    analysis_df['recommendation'].isin(['强烈推荐', '推荐'])
].sort_values('composite_score', ascending=False).head(10)

# 创建交互式推荐表格（带颜色编码 + 悬停详情）
fig = go.Figure(data=[go.Table(
    header=dict(
        values=[
            '<b>排名</b>', '<b>代码</b>', '<b>名称</b>', '<b>板块</b>',
            '<b>盈亏比</b>', '<b>评分</b>', '<b>建议</b>'
        ],
        fill_color='royalblue',
        align='center',
        font=dict(color='white', size=11)
    ),
    cells=dict(
        values=[
            range(1, len(top_recommendations)+1),
            top_recommendations['code'],
            top_recommendations['name'],
            top_recommendations['sector'],
            top_recommendations['profit_loss_ratio'].apply(lambda x: f'{x:.1f}x'),
            top_recommendations['fundamental_score'].apply(lambda x: f'{x:.0f}'),
            top_recommendations['recommendation']
        ],
        fill_color=[
            ['lightgreen' if r=='强烈推荐' else 'lightblue' if r=='推荐' else 'white'
             for r in top_recommendations['recommendation']]
        ] * 7,
        align='center',
        font=dict(size=10),
        height=30
    )
)])

fig.update_layout(
    title='🏆 推荐标的排行榜（交互式）',
    height=400,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

# 导出为交互式推荐报告（可选）
# fig.write_html('output/top_recommendations.html', include_plotlyjs='cdn')

## 📊 分析总结 + 导出交互式报告

In [ ]:
# 创建交互式分析总结仪表板（Plotly Dashboard 风格）
summary_metrics = pd.DataFrame([
    {'指标': '总标的数', '数值': len(analysis_df), '单位': '只'},
    {'指标': '推荐标的', '数值': len(analysis_df[analysis_df['recommendation'].isin(['强烈推荐', '推荐'])]), '单位': '只'},
    {'指标': '平均盈亏比', '数值': analysis_df['profit_loss_ratio'].mean(), '单位': 'x'},
    {'指标': '平均评分', '数值': analysis_df['fundamental_score'].mean(), '单位': '分'},
    {'指标': '板块数量', '数值': analysis_df['sector'].nunique(), '单位': '个'},
    {'指标': '推荐占比', '数值': (analysis_df['recommendation'].isin(['强烈推荐', '推荐'])).mean()*100, '单位': '%'}
])

# 创建交互式总结仪表板（多图表组合）
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['分析概览', '关键指标'],
    specs=[[{'type': 'table'}, {'type': 'bar'}]]
)

# 左侧：分析概览表格（交互式）
fig.add_trace(go.Table(
    header=dict(
        values=[f'<b>{col}</b>' for col in ['指标', '数值', '单位']],
        fill_color='royalblue',
        align='center',
        font=dict(color='white', size=11)
    ),
    cells=dict(
        values=[summary_metrics[col] for col in ['指标', '数值', '单位']],
        align='center',
        font=dict(size=10),
        height=25
    )
), row=1, col=1)

# 右侧：关键指标对比条形图（交互式）
fig.add_trace(go.Bar(
    x=summary_metrics['指标'],
    y=summary_metrics['数值'],
    name='指标值',
    marker_color='skyblue',
    text=[f"{v}{u}" for v, u in zip(summary_metrics['数值'], summary_metrics['单位'])],
    textposition='auto'
), row=1, col=2)

fig.update_layout(
    title='📋 可视化分析总结',
    height=350,
    margin=dict(l=20, r=20, t=50, b=20),
    showlegend=False
)

fig.update_xaxes(title_text='指标', row=1, col=2)
fig.update_yaxes(title_text='数值', row=1, col=2)

fig.show()

# 导出为交互式分析报告（可选）
# fig.write_html('output/analysis_report.html', include_plotlyjs='cdn')

In [ ]:
# 最终状态输出 + Plotly 高级功能提示 + 导出指南 + 分析建议
print("\n" + "="*60)
print("🎉 可视化分析完成！")
print("📊 所有图表均为 Plotly 交互式可视化")
print("💡 高级功能提示：")
print("   • 悬停查看标的详情 + 计算参数")
print("   • 拖动缩放查看净值曲线细节")
print("   • 双击图例筛选特定板块/建议")
print("   • 使用风险矩阵筛选高风险标的")
print("   • 点击表格列标题排序数据")
print("\n📤 导出指南：")
print("   • PNG/SVG: 点击右上角相机图标")
print("   • HTML: fig.write_html('report.html')")
print("   • 交互式 Dashboard: 结合 Plotly Dash 部署")
print("\n🔍 分析建议：")
print("   • 重点关注盈亏比>2.5 且评分>70 的标的")
print("   • 控制单一板块仓位不超过 30%")
print("   • 定期复盘风险矩阵，调整持仓结构")
print("="*60)